# ASS-ADE LoRA training on Kaggle Notebooks (free P100/T4)

**Kaggle gives every account 30 hours per week of P100 or T4 GPU.** Way more than we need for a LoRA fine-tune on ≤5k samples.

## Setup (one-time, 5 minutes)

1. Fork / create a Kaggle notebook (kaggle.com → Code → New Notebook)
2. In the notebook **Settings panel**:
   - Accelerator: **GPU P100** (or T4 x2 for bigger models)
   - Internet: **On**
   - Persistence: **Variables and files**
3. Add these **Secrets** (🔒 icon in the right sidebar):
   - `AAAA_NEXUS_OWNER_TOKEN` — storefront owner token (owner-only sample export)
   - `HF_TOKEN` — HuggingFace write token (for adapter upload)
4. Schedule: open the "Schedule notebook" menu → weekly @ Sunday 06:00 UTC
5. Save + run once to verify, then let the schedule take over.

In [ ]:
# Install ass-ade with LoRA extras
!pip install -q 'git+https://github.com/atomadictech/ass-ade.git#egg=ass-ade[lora]'

In [ ]:
# Pull secrets from Kaggle's secrets store
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['AAAA_NEXUS_OWNER_TOKEN'] = secrets.get_secret('AAAA_NEXUS_OWNER_TOKEN')
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['AAAA_NEXUS_BASE_URL'] = 'https://atomadic.tech'

In [ ]:
# Confirm GPU is visible
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Train on each language pool that has enough samples
import subprocess, sys

# Change this list to the languages you want to train for this week
LANGUAGES = ['python', 'rust', 'typescript']

# With P100 or T4, `medium` (Qwen2.5-Coder-7B) is the sweet spot.
# For T4 x2 (29GB total), try --profile large.
PROFILE = 'medium'
EPOCHS = 3
MAX_SAMPLES = 5000

for lang in LANGUAGES:
    print(f'\n===== Training {lang} ({PROFILE}) =====')
    rc = subprocess.call([
        sys.executable, '-m', 'scripts.lora_train',
        '--lang', lang,
        '--profile', PROFILE,
        '--epochs', str(EPOCHS),
        '--max-samples', str(MAX_SAMPLES),
        '--min-samples', '20',
        '--upload', 'hf',
        '--hf-repo', f'atomadic/lora-{lang}',
        '--storefront', 'https://atomadic.tech',
    ])
    print(f'{lang} exit code: {rc}')

In [ ]:
# Verify promotion landed on the storefront
import httpx
for lang in LANGUAGES:
    r = httpx.get(f'https://atomadic.tech/v1/lora/adapter/{lang}', timeout=10.0)
    print(f'{lang}: {r.json()}')